In [1]:
pip install dash sqlalchemy pymysql pandas plotly

In [2]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

##DB Credentials
db_user = 'root'
db_password = quote_plus('your_password')
db_host = 'localhost'
db_name = 'neha'

##Create Connection Engine
engine = create_engine(f'mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}')

## Query Data
query = "SELECT * from flights"
df = pd.read_sql(query, engine)
df.head()

In [3]:
## SQL Analysis Queries
from sqlalchemy import text

conn = engine.connect()

# Q1 - Total flights per airline
q1 = pd.read_sql(text("""
    SELECT airline, COUNT(*) AS total_flights
    FROM flights
    GROUP BY airline
    ORDER BY total_flights DESC
"""), conn)
q1

In [4]:
# Q2 - Average occupancy rate per airline
q2 = pd.read_sql(text("""
    SELECT airline, ROUND(AVG(occupancy_rate), 2) AS avg_occupancy
    FROM flights
    GROUP BY airline
    ORDER BY avg_occupancy DESC
"""), conn)
q2

In [5]:
# Q3 - Top 5 busiest routes
q3 = pd.read_sql(text("""
    SELECT source, destination, COUNT(*) AS total_flights
    FROM flights
    GROUP BY source, destination
    ORDER BY total_flights DESC
    LIMIT 5
"""), conn)
q3

In [6]:
# Q4 - Monthly flight count and average occupancy
q4 = pd.read_sql(text("""
    SELECT month, COUNT(*) AS total_flights,
           ROUND(AVG(occupancy_rate), 2) AS avg_occupancy
    FROM flights
    GROUP BY month
    ORDER BY total_flights DESC
"""), conn)
q4

In [7]:
# Q5 - Total revenue per airline
q5 = pd.read_sql(text("""
    SELECT airline, SUM(revenue) AS total_revenue
    FROM flights
    GROUP BY airline
    ORDER BY total_revenue DESC
"""), conn)
q5

In [8]:
# Q6 - Average delay per airline
q6 = pd.read_sql(text("""
    SELECT airline, ROUND(AVG(delay_minutes), 2) AS avg_delay
    FROM flights
    GROUP BY airline
    ORDER BY avg_delay DESC
"""), conn)
q6

In [9]:
# Q7 - Cancellation rate per airline
q7 = pd.read_sql(text("""
    SELECT airline, COUNT(*) AS total_flights,
           SUM(is_cancelled) AS cancelled,
           ROUND(SUM(is_cancelled) * 100.0 / COUNT(*), 2) AS cancellation_rate
    FROM flights
    GROUP BY airline
    ORDER BY cancellation_rate DESC
"""), conn)
q7

In [10]:
# Q8 - Window Function: Revenue rank per route
q8 = pd.read_sql(text("""
    SELECT airline, source, destination,
           SUM(revenue) AS route_revenue,
           RANK() OVER (PARTITION BY source ORDER BY SUM(revenue) DESC) AS revenue_rank
    FROM flights
    GROUP BY airline, source, destination
    ORDER BY source, revenue_rank
    LIMIT 15
"""), conn)
q8

In [11]:
# Q9 - CTE: Flights above average occupancy
q9 = pd.read_sql(text("""
    WITH avg_occ AS (
        SELECT AVG(occupancy_rate) AS overall_avg FROM flights
    )
    SELECT airline, source, destination, occupancy_rate
    FROM flights, avg_occ
    WHERE occupancy_rate > overall_avg
    ORDER BY occupancy_rate DESC
    LIMIT 10
"""), conn)
q9

In [12]:
# Q10 - Underperforming routes
q10 = pd.read_sql(text("""
    SELECT source, destination,
           ROUND(AVG(occupancy_rate), 2) AS avg_occupancy,
           SUM(revenue) AS total_revenue
    FROM flights
    GROUP BY source, destination
    HAVING avg_occupancy < 55
    ORDER BY avg_occupancy ASC
    LIMIT 10
"""), conn)
q10

In [13]:
conn.close()

In [14]:
## Create Dash Application with Plotly
from dash import Dash, dcc, html
import plotly.express as px

##Create Dash App
app = Dash(__name__)

##Plot using plotly
fig_scatter = px.scatter(df, x='ticket_price', y='occupancy_rate', color='airline',
                         title='Ticket Price vs Occupancy Rate',
                         labels={'ticket_price': 'Ticket Price (Rs)', 'occupancy_rate': 'Occupancy %'})

fig_bar = px.bar(df, x='airline', y='passengers', color='airline',
                 title='Total Passengers per Airline',
                 labels={'passengers': 'Passengers', 'airline': 'Airline'})

fig_pie = px.pie(df, names='airline', values='revenue',
                 title='Revenue Share per Airline')

fig_line = px.line(df.groupby('month')['passengers'].sum().reset_index(),
                   x='month', y='passengers',
                   title='Monthly Passenger Trend',
                   labels={'passengers': 'Total Passengers', 'month': 'Month'})

app.layout = html.Div(children=[
    html.H1(children='Airline Operations Analytics Dashboard'),
    dcc.Graph(id='scatter-graph', figure=fig_scatter),
    dcc.Graph(id='bar-chart', figure=fig_bar),
    dcc.Graph(id='pie-chart', figure=fig_pie),
    dcc.Graph(id='line-chart', figure=fig_line),
    html.Div(children='Dashboard for visualization of airline operations data')
])

if __name__ == '__main__':
    app.run(debug=True, jupyter_mode='inline')

In [15]:
## Interactive Dashboard with Filters
from dash import dcc, html, Input, Output
import dash
import plotly.express as px
import pandas as pd

# Reuse the same data (df) from the previous cell

# Create the Dash App
app = dash.Dash(__name__)

# Layout with Dropdown
app.layout = html.Div([
    html.H1("Airline Operations Analytics"),

    # Dropdown for selecting Airline
    dcc.Dropdown(
        id='airline-dropdown',
        options=[{'label': airline, 'value': airline} for airline in df['airline'].unique()],
        placeholder="Select an Airline",
        multi=True
    ),

    # Dropdown for selecting Month
    dcc.Dropdown(
        id='month-dropdown',
        options=[{'label': month, 'value': month} for month in df['month'].unique()],
        placeholder="Select a Month",
        multi=True
    ),

    # Graphs
    dcc.Graph(id='filtered-scatter'),
    dcc.Graph(id='filtered-bar'),
    dcc.Graph(id='filtered-occupancy')
])

# Callback for interactivity
@app.callback(
    [Output('filtered-scatter', 'figure'),
     Output('filtered-bar', 'figure'),
     Output('filtered-occupancy', 'figure')],
    [Input('airline-dropdown', 'value'),
     Input('month-dropdown', 'value')]
)
def update_graphs(selected_airlines, selected_months):
    filtered_df = df.copy()
    if selected_airlines:
        filtered_df = filtered_df[filtered_df['airline'].isin(selected_airlines)]
    if selected_months:
        filtered_df = filtered_df[filtered_df['month'].isin(selected_months)]

    # Scatter plot
    scatter_fig = px.scatter(filtered_df, x='ticket_price', y='occupancy_rate',
                             color='airline', title='Ticket Price vs Occupancy Rate')

    # Bar chart
    bar_fig = px.bar(filtered_df, x='airline', y='revenue',
                     color='airline', title='Revenue per Airline')

    # Occupancy line
    occ_fig = px.line(filtered_df.groupby('month')['occupancy_rate'].mean().reset_index(),
                      x='month', y='occupancy_rate', title='Monthly Avg Occupancy Rate')

    return scatter_fig, bar_fig, occ_fig

if __name__ == '__main__':
    app.run(debug=True, jupyter_mode='inline')